Folder Hierarchy Analysis Tool

This script recursively analyzes a directory structure and generates a detailed CSV report containing:
- Complete folder paths and names
- Unique hierarchical IDs (1.2.3 format)
- Ancestry relationships between folders
- Depth and level information

Features:
- Recursively walks directory trees
- Builds a long-format DataFrame with folder ancestry
- Assigns unique hierarchical identifiers (1.2.4-style)
- Outputs results to folder_hierarchy.csv

Output Columns:
- folder_path: Full path of the folder
- folder_name: Last part of the path (folder's name)
- unique_id: Hierarchical ID (e.g., 1.2.1)
- depth_from_root: Levels below the given root
- ancestor_path: Path to an ancestor of the folder
- ancestor_name: Name of the ancestor folder
- ancestor_id: Unique ID of the ancestor folder
- ancestor_level: How many steps above the folder (0 = parent)
- depth_from_ancestor: Distance below that ancestor

Usage:
    python folder_hierarchy.py /path/to/your/folder

Requires:
    - pathlib
    - pandas


In [ ]:
import os
from pathlib import Path
import pandas as pd
from collections import defaultdict

def build_folder_hierarchy(root_path):
    root = Path(root_path).resolve()
    all_folders = sorted([p for p in root.rglob("*") if p.is_dir()])
    all_folders.insert(0, root)  # include the root
    
    # Track folder IDs and sibling counters
    folder_id_map = {}  # maps folder path → hierarchical ID 
    child_counters = defaultdict(int)  # maps parent path → child index
    
    # Track ancestry records for long-format output
    records = []
    
    # Assign IDs in traversal order
    for folder in all_folders:
        folder_str = str(folder)
        parent = folder.parent
        parent_str = str(parent)
        
        # Root folder: assign top-level ID
        if folder == root:
            folder_id = "1"
        else:
            parent_id = folder_id_map.get(parent_str, "X")  # fallback if parent not yet assigned
            child_counters[parent_str] += 1
            index = child_counters[parent_str]
            folder_id = f"{parent_id}.{index}"
            
        folder_id_map[folder_str] = folder_id

    # Construct long-format ancestry DataFrame
    for folder in all_folders:
        folder_path = str(folder)
        folder_name = folder.name
        depth_from_root = len(folder.relative_to(root).parts)
        unique_id = folder_id_map[folder_path]
        
        ancestors = list(folder.parents)
        ancestors_in_tree = [a for a in ancestors if str(a).startswith(str(root))]
        
        for level, ancestor in enumerate(reversed(ancestors_in_tree)):
            ancestor_path = str(ancestor)
            ancestor_name = ancestor.name
            ancestor_id = folder_id_map.get(ancestor_path, "")
            depth_from_ancestor = depth_from_root - len(Path(ancestor_path).relative_to(root).parts)
            
            records.append({
                "folder_path": folder_path,
                "folder_name": folder_name,
                "unique_id": unique_id,
                "depth_from_root": depth_from_root,
                "ancestor_path": ancestor_path,
                "ancestor_name": ancestor_name,
                "ancestor_id": ancestor_id,
                "ancestor_level": level,
                "depth_from_ancestor": depth_from_ancestor
            })
            
    return pd.DataFrame(records)

if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description="Build a folder hierarchy with unique IDs and ancestry tracking.")
    parser.add_argument("root_folder", help="Path to the root folder to scan")
    args = parser.parse_args()
    
    df = build_folder_hierarchy(args.root_folder)
    output_file = "folder_hierarchy.csv"
    df.to_csv(output_file, index=False)
    print(f"Hierarchy written to {output_file}")
